# Wave Equation — Physics-Informed Neural Network

The **wave equation** describes the propagation of mechanical waves (sound, vibrations,
water waves) through a medium. Unlike Burgers' equation, it is second-order in both
space *and* time, which requires two initial conditions.

## The Equation

$$
\frac{\partial^2 u}{\partial t^2} = c^2\,\frac{\partial^2 u}{\partial x^2}
$$

where:

- $u(x, t)$ — displacement field (e.g. transverse displacement of a vibrating string)
- $c > 0$ — wave propagation speed

## Problem Setup

| Component | Expression |
|---|---|
| Domain | $x \in [0,1],\; t \in [0,1]$ |
| Wave speed | $c = 1$ |
| Left BC | $u(0, t) = 0$ |
| Right BC | $u(1, t) = 0$ |
| Displacement IC | $u(x, 0) = \sin(\pi x)$ |
| Velocity IC | $u_t(x, 0) = 0$ |

## Exact Solution

With these boundary and initial conditions the exact solution is:

$$
u(x, t) = \sin(\pi x)\,\cos(\pi c\,t)
$$

This represents a **standing wave**: the spatial mode $\sin(\pi x)$ oscillates in time
with angular frequency $\omega = \pi c$. Having an exact solution makes this a clean
benchmark for validating the PINN approach.

## Key Differences from Burgers' Equation

The wave equation is **second-order in time**, which introduces two extra considerations
for the PINN:

1. **Two initial conditions** — one for displacement $u(x, 0)$ and one for velocity
   $u_t(x, 0)$.  The velocity IC cannot be encoded as a simple label; it must be
   computed via automatic differentiation.
2. **Computing $u_{tt}$** — requires two sequential applications of
   `torch.autograd.grad`, one for $u_t$ and a second for $\partial_t(u_t)$.
3. **Three-term loss** — the total PINN loss combines a data term (BCs + displacement
   IC), a PDE residual term, and a velocity IC term.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from tqdm import tqdm

In [ ]:
class WaveNet(nn.Module):
    """Fully-connected network approximating u(x, t) for the wave equation.

    Architecture: 2 → 32 → 32 → 32 → 32 → 1, Tanh activations throughout.
    A slightly wider network than BurgersNet because the exact solution
    sin(πx)cos(πt) requires representing oscillatory behaviour in both the
    spatial and temporal dimensions simultaneously.
    """

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32),  nn.Tanh(),
            nn.Linear(32, 32), nn.Tanh(),
            nn.Linear(32, 32), nn.Tanh(),
            nn.Linear(32, 32), nn.Tanh(),
            nn.Linear(32, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

In [ ]:
class WavePINN:
    """Physics-Informed Neural Network solver for the 1D wave equation.

    PDE:  u_tt = c²·u_xx,  x ∈ [0, 1],  t ∈ [0, 1]
    BCs:  u(0, t) = u(1, t) = 0
    ICs:  u(x, 0) = sin(πx),  u_t(x, 0) = 0

    Exact solution:  u(x, t) = sin(πx)·cos(πct)

    Total loss = loss_data + loss_pde + loss_vel, where:
      loss_data — MSE on BC + IC displacement labels.
      loss_pde  — MSE of the PDE residual u_tt − c²u_xx at collocation points.
      loss_vel  — MSE of u_t(x, 0) against zero (velocity IC, via autograd).
    """

    C: float = 1.0  # wave propagation speed

    def __init__(self, h: float = 0.05, k: float = 0.05):
        """
        Args:
            h: Spatial grid spacing for collocation points.
            k: Temporal grid spacing for collocation points.
        """
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = WaveNet().to(self.device)
        self.criterion = nn.MSELoss()
        self.iter = 0

        x = torch.linspace(0.0, 1.0, round(1.0 / h) + 1)
        t = torch.linspace(0.0, 1.0, round(1.0 / k) + 1)

        # --- Collocation grid (PDE residual enforced at every node) ---
        X_grid, T_grid = torch.meshgrid(x, t, indexing="ij")
        self.X_col = torch.stack([X_grid.ravel(), T_grid.ravel()], dim=1).to(self.device)
        self.X_col.requires_grad_(True)

        # --- IC velocity points (need autograd to compute u_t at t=0) ---
        # Kept separate from X_col so their gradient tape is isolated.
        X_ic, T_ic = torch.meshgrid(x, t[:1], indexing="ij")
        self.X_ic_vel = torch.stack([X_ic.ravel(), T_ic.ravel()], dim=1).to(self.device)
        self.X_ic_vel.requires_grad_(True)

        # --- Labeled training data: BCs + IC displacement ---
        self.X_train, self.y_train = self._build_training_data(x, t)
        self.X_train = self.X_train.to(self.device)
        self.y_train = self.y_train.to(self.device)

        # --- Optimizers ---
        self.optimizer_adam = torch.optim.Adam(self.model.parameters())
        self.optimizer_lbfgs = torch.optim.LBFGS(
            self.model.parameters(),
            max_iter=50_000,
            max_eval=50_000,
            tolerance_grad=1e-7,
            tolerance_change=float(np.finfo(float).eps),
            line_search_fn="strong_wolfe",
            history_size=50,
        )

    # ------------------------------------------------------------------
    # Private helpers
    # ------------------------------------------------------------------

    def _build_training_data(
        self, x: torch.Tensor, t: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """Assemble labeled (X, y) pairs for BCs and IC displacement."""

        # Left BC: u(0, t) = 0
        X_l, T_l = torch.meshgrid(x[:1], t, indexing="ij")
        bc_left = torch.stack([X_l.ravel(), T_l.ravel()], dim=1)
        y_left  = torch.zeros(len(bc_left), 1)

        # Right BC: u(1, t) = 0
        X_r, T_r = torch.meshgrid(x[-1:], t, indexing="ij")
        bc_right = torch.stack([X_r.ravel(), T_r.ravel()], dim=1)
        y_right  = torch.zeros(len(bc_right), 1)

        # IC displacement: u(x, 0) = sin(πx)
        X_ic, T_ic = torch.meshgrid(x, t[:1], indexing="ij")
        ic   = torch.stack([X_ic.ravel(), T_ic.ravel()], dim=1)
        y_ic = torch.sin(torch.pi * ic[:, 0]).unsqueeze(1)

        return torch.cat([bc_left, bc_right, ic]), torch.cat([y_left, y_right, y_ic])

    def _loss(self) -> torch.Tensor:
        """Compute total loss = data loss + PDE residual loss + velocity IC loss.

        Serves as the closure for both Adam and L-BFGS. Zeroes gradients,
        performs forward passes, computes all three loss components, runs
        backpropagation, and returns the scalar loss.
        """
        self.optimizer_adam.zero_grad()
        self.optimizer_lbfgs.zero_grad()

        # --- Data loss: BCs + IC displacement ---
        y_pred    = self.model(self.X_train)
        loss_data = self.criterion(y_pred, self.y_train)

        # --- PDE residual: u_tt − c²·u_xx = 0 ---
        u = self.model(self.X_col)  # (N_col, 1)

        # First derivatives: ∂u/∂x and ∂u/∂t
        grad_1 = torch.autograd.grad(
            u, self.X_col,
            grad_outputs=torch.ones_like(u),
            create_graph=True, retain_graph=True,
        )[0]
        u_x = grad_1[:, 0]  # ∂u/∂x
        u_t = grad_1[:, 1]  # ∂u/∂t

        # Second spatial derivative: ∂²u/∂x²
        grad_2x = torch.autograd.grad(
            u_x, self.X_col,
            grad_outputs=torch.ones_like(u_x),
            create_graph=True, retain_graph=True,
        )[0]
        u_xx = grad_2x[:, 0]  # ∂²u/∂x²

        # Second time derivative: ∂²u/∂t² (requires a second grad pass through u_t)
        grad_2t = torch.autograd.grad(
            u_t, self.X_col,
            grad_outputs=torch.ones_like(u_t),
            create_graph=True, retain_graph=True,
        )[0]
        u_tt = grad_2t[:, 1]  # ∂²u/∂t²

        # Wave equation residual: u_tt − c²·u_xx = 0
        residual = u_tt - self.C ** 2 * u_xx
        loss_pde = self.criterion(residual, torch.zeros_like(residual))

        # --- Velocity IC: u_t(x, 0) = 0 ---
        # X_ic_vel has requires_grad=True so autograd can differentiate w.r.t. t.
        u_ic = self.model(self.X_ic_vel)
        grad_ic = torch.autograd.grad(
            u_ic, self.X_ic_vel,
            grad_outputs=torch.ones_like(u_ic),
            create_graph=True, retain_graph=True,
        )[0]
        u_t_ic   = grad_ic[:, 1]  # ∂u/∂t evaluated at t=0
        loss_vel = self.criterion(u_t_ic, torch.zeros_like(u_t_ic))

        loss = loss_data + loss_pde + loss_vel
        loss.backward()

        self.iter += 1
        if self.iter % 100 == 0:
            print(f"  iter {self.iter:>6d}  loss {loss.item():.6e}")

        return loss

    # ------------------------------------------------------------------
    # Public interface
    # ------------------------------------------------------------------

    def train(self, adam_steps: int = 2000) -> None:
        """Two-phase training: Adam pre-conditioning then L-BFGS fine-tuning."""
        self.model.train()

        print("Phase 1 — Adam optimizer")
        for _ in tqdm(range(adam_steps), desc="Adam"):
            self._loss()
            self.optimizer_adam.step()

        print("\nPhase 2 — L-BFGS optimizer")
        self.optimizer_lbfgs.step(self._loss)
        print("Done.")

    def predict(self, X: torch.Tensor) -> np.ndarray:
        """Evaluate the trained network on a set of (x, t) points.

        Args:
            X: Tensor of shape (N, 2) with columns [x, t].

        Returns:
            NumPy array of shape (N, 1) with predicted u values.
        """
        self.model.eval()
        with torch.no_grad():
            return self.model(X.to(self.device)).cpu().numpy()

In [ ]:
pinn = WavePINN(h=0.05, k=0.05)
pinn.train(adam_steps=2000)

In [ ]:
# --- Evaluate PINN and exact solution on a fine grid ---
nx, nt = 300, 150
x_plot = torch.linspace(0.0, 1.0, nx)
t_plot = torch.linspace(0.0, 1.0, nt)

X_grid, T_grid = torch.meshgrid(x_plot, t_plot, indexing="ij")
X_eval = torch.stack([X_grid.ravel(), T_grid.ravel()], dim=1)

u_pred  = pinn.predict(X_eval).reshape(nx, nt)
u_exact = (torch.sin(torch.pi * X_grid) * torch.cos(torch.pi * T_grid)).numpy()
abs_err = np.abs(u_pred - u_exact)

# --- Space-time heatmaps: prediction | exact solution | absolute error ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

im0 = axes[0].contourf(t_plot, x_plot, u_pred,  levels=100, cmap="RdBu_r")
fig.colorbar(im0, ax=axes[0], label="u(x, t)")
axes[0].set_xlabel("t"); axes[0].set_ylabel("x")
axes[0].set_title("PINN Prediction")

im1 = axes[1].contourf(t_plot, x_plot, u_exact, levels=100, cmap="RdBu_r")
fig.colorbar(im1, ax=axes[1], label="u(x, t)")
axes[1].set_xlabel("t"); axes[1].set_ylabel("x")
axes[1].set_title(r"Exact: $\sin(\pi x)\cos(\pi t)$")

im2 = axes[2].contourf(t_plot, x_plot, abs_err, levels=100, cmap="hot_r")
fig.colorbar(im2, ax=axes[2], label="|error|")
axes[2].set_xlabel("t"); axes[2].set_ylabel("x")
axes[2].set_title(f"Absolute Error  (max = {abs_err.max():.2e})")

plt.tight_layout()
plt.show()

# --- Snapshot comparison at selected time slices ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, t_snap in zip(axes, [0.0, 0.5, 1.0]):
    idx = int(t_snap * (nt - 1))
    ax.plot(x_plot, u_exact[:, idx], "k--", label="Exact", linewidth=2)
    ax.plot(x_plot, u_pred[:, idx],  "b-",  label="PINN",  linewidth=1.5)
    ax.set_xlabel("x")
    ax.set_ylabel("u(x, t)")
    ax.set_title(f"t = {t_snap:.1f}")
    ax.legend()
    ax.grid(True, linestyle="--", alpha=0.5)

plt.suptitle("Wave equation — solution snapshots", y=1.02)
plt.tight_layout()
plt.show()